# 05 - Google Reanalysis vs Floodscan

For a single LGA, computes lag correlations between each individual upstream gauge's streamflow (Google GRRR reanalysis) and Floodscan SFED flood fraction.
Results are shown as a heatmap (gauges × lag days).

**Inputs (from Azure):**
- `ds-aa-nga-flooding/processed/floodscan/floodscan_hfr_lgas.parquet`
- `ds-aa-nga-flooding/processed/sel_lga_upstream_gauges.parquet`
- Google GRRR reanalysis zarr (public GCS bucket)

**No datasets saved to Azure.**

In [307]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [308]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import ocha_stratus as stratus
from dotenv import load_dotenv

from src.datasources import grrr

load_dotenv()

True

## Load data

In [310]:
# Floodscan - SFED band only
df_fs_all = stratus.load_parquet_from_blob(
    "ds-aa-nga-flooding/processed/floodscan/floodscan_hfr_lgas.parquet"
)
df_fs = df_fs_all[df_fs_all["band"] == "SFED"].copy()
df_fs["valid_date"] = pd.to_datetime(df_fs["valid_date"])

# Upstream gauge relationships - quality-verified only
df_upstream_all = stratus.load_parquet_from_blob(
    "ds-aa-nga-flooding/processed/sel_lga_upstream_gauges.parquet"
)
df_upstream = df_upstream_all[df_upstream_all["quality_verified"] == True].copy()

print(f"Floodscan: {df_fs['pcode'].nunique()} LGAs, {df_fs.valid_date.min().date()} – {df_fs.valid_date.max().date()}")
print(f"Upstream gauges (QV only): {df_upstream.lga_pcode.nunique()} LGAs, {len(df_upstream)} gauge-LGA pairs")

Floodscan: 42 LGAs, 1998-01-12 – 2026-02-15
Upstream gauges (QV only): 23 LGAs, 294 gauge-LGA pairs


### Available LGAs

LGAs that have both quality-verified upstream gauges and Floodscan coverage, ranked by flood signal.

In [345]:
lgas_both = set(df_upstream["lga_pcode"]) & set(df_fs["pcode"])

gauge_counts = (
    df_upstream[df_upstream["lga_pcode"].isin(lgas_both)]
    .groupby(["lga_pcode", "lga_name"])["gauge_id"]
    .nunique()
    .reset_index()
    .rename(columns={"gauge_id": "n_qv_gauges"})
)
peak_sfed = (
    df_fs[df_fs["pcode"].isin(lgas_both)]
    .groupby("pcode")["mean"]
    .quantile(0.99)
    .reset_index()
    .rename(columns={"pcode": "lga_pcode", "mean": "sfed_p99"})
)
df_lga_summary = (
    gauge_counts.merge(peak_sfed, on="lga_pcode")
    .sort_values("sfed_p99", ascending=False)
    .reset_index(drop=True)
)
df_lga_summary

,lga_pcode,lga_name,n_qv_gauges,sfed_p99
0,NG023006,Ibaji,19,0.337760
1,NG027017,Mokwa,4,0.144671
2,NG035006,Ibi,7,0.135879
3,NG023011,Kogi,17,0.117587
4,NG035010,Lau,2,0.109021
5,NG035005,Gassol,7,0.104287
6,NG035008,Karim-Lamido,4,0.087338
7,NG023012,Lokoja,17,0.078719
8,NG006003,Kolokuma/Opokuma,21,0.078326
9,NG023004,Bassa,17,0.075163


## Parameters

Set the LGA pcode to analyse. Run the *Available LGAs* cell below first if you need to find a pcode.

In [ ]:
FOCUS_PCODE = "NG035005"      # LGA pcode — verify from Available LGAs table above
MIN_LAG = -5                   # negative = SFED leads streamflow (gauge within or near LGA)
MAX_LAG = 30                   # positive = streamflow leads SFED (gauge upstream)
FLOOD_SEASON_MONTHS = [7, 8, 9, 10, 11]  # July–November
MIN_SFED = 0.02                # exclude trivially small SFED values (0–1 scale)
FLOODSCAN_VAR = "mean"          # Floodscan column to use: "mean" or "max"

## Load reanalysis streamflow for focus LGA

In [102]:
valid_pcodes = set(df_upstream["lga_pcode"]) & set(df_fs["pcode"])
if FOCUS_PCODE not in valid_pcodes:
    raise ValueError(
        f"{FOCUS_PCODE!r} has no quality-verified upstream gauges with Floodscan coverage. "
        f"Choose a pcode from the table above."
    )

lga_name = df_upstream[df_upstream["lga_pcode"] == FOCUS_PCODE]["lga_name"].iloc[0]

# Sort by river first, then hop_distance — groups tributaries together
df_lga_upstream = (
    df_upstream[df_upstream["lga_pcode"] == FOCUS_PCODE]
    .sort_values(["gauge_river", "hop_distance", "gauge_id"])
    .copy()
)
gauge_ids = df_lga_upstream["gauge_id"].tolist()

print(f"LGA: {lga_name} ({FOCUS_PCODE})")
print(f"QV upstream gauges: {len(gauge_ids)}")
print(f"Hop distance range: {df_lga_upstream.hop_distance.min()} – {df_lga_upstream.hop_distance.max()}")

ds_ra = grrr.load_reanalysis(gauge=gauge_ids)
df_ra = grrr.process_reanalysis(ds_ra)
df_ra["valid_date"] = df_ra["valid_time"].dt.normalize()
print(f"Reanalysis: {df_ra.valid_date.min().date()} – {df_ra.valid_date.max().date()}")

LGA: Gassol (NG035005)
QV upstream gauges: 7
Hop distance range: 0 – 30
Reanalysis: 1980-01-01 – 2023-12-23


## Align each gauge with Floodscan SFED

In [ ]:
df_sfed = (
    df_fs[df_fs["pcode"] == FOCUS_PCODE][["valid_date", FLOODSCAN_VAR]]
    .rename(columns={FLOODSCAN_VAR: "sfed"})
)

# Align each gauge individually, restrict to wet season and meaningful SFED values
gauge_dfs = {}
for gauge_id in gauge_ids:
    df_g = df_ra[df_ra["gauge_id"] == gauge_id][["valid_date", "streamflow"]].copy()
    df_merged = (
        df_sfed.merge(df_g, on="valid_date", how="inner")
        .sort_values("valid_date")
        .reset_index(drop=True)
    )
    df_merged = df_merged[df_merged["valid_date"].dt.month.isin(FLOOD_SEASON_MONTHS)]
    df_merged = df_merged[df_merged["sfed"] >= MIN_SFED]
    gauge_dfs[gauge_id] = df_merged

sample = next(iter(gauge_dfs.values()))
n_years = sample["valid_date"].dt.year.nunique()
print(f"{len(gauge_dfs)} gauges aligned")
print(f"Wet-season days with SFED {FLOODSCAN_VAR} ≥ {MIN_SFED}: {len(sample)} ({n_years} years, months {FLOOD_SEASON_MONTHS})")

## Lag correlation heatmap

Spearman r between each upstream gauge's streamflow (shifted by lag days) and LGA Floodscan SFED (max pixel value per LGA).

In [104]:
lags = list(range(MIN_LAG, MAX_LAG + 1))

# Build correlation matrix: rows = gauges (in hop-distance order), columns = lags
# Spearman rank correlation — captures monotonic relationships and is robust to
# the zero-inflation in both signals (many low-streamflow / near-zero SFED days).
corr_rows = {}
for gauge_id in gauge_ids:  # already sorted by hop_distance
    df_g = gauge_dfs[gauge_id]
    corr_rows[gauge_id] = [
        df_g["sfed"].corr(df_g["streamflow"].shift(lag), method="spearman")
        for lag in lags
    ]

df_corr = pd.DataFrame(corr_rows, index=lags).T
df_corr.index.name = "gauge_id"
df_corr.columns.name = "lag_days"

# Row labels: hop distance + river + short gauge ID
def gauge_label(gauge_id):
    row = df_lga_upstream[df_lga_upstream["gauge_id"] == gauge_id].iloc[0]
    short_id = gauge_id.replace("hybas_", "")
    return f"[{int(row.hop_distance)}] {row.gauge_river} ({short_id})"

labels = [gauge_label(g) for g in df_corr.index]

# Best lag per gauge
best = df_corr.idxmax(axis=1)
print("Best lag per gauge (sorted by hop distance):")
for gauge_id, best_lag in best.items():
    print(f"  {gauge_label(gauge_id)}: lag={best_lag}d, r={df_corr.loc[gauge_id, best_lag]:.3f}")

Best lag per gauge (sorted by hop distance):
  [0] Benue (1120872430): lag=-4d, r=0.610
  [0] Benue (1120887720): lag=-5d, r=0.588
  [1] Benue (1120903830): lag=-2d, r=0.457
  [2] Benue (1120865420): lag=-4d, r=0.609
  [5] Benue (1120856330): lag=-3d, r=0.607
  [12] Benue (1120846650): lag=-3d, r=0.604
  [30] Benue (1120842550): lag=-1d, r=0.611


In [ ]:
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe

# Color per river for y-axis labels
RIVER_COLORS = {
    "Niger": "#007CE0",
    "Benue": "#F2645A",
}
DEFAULT_RIVER_COLOR = "#666666"

n_gauges = len(df_corr)
fig_h = max(4, n_gauges * 0.45 + 2)
fig, ax = plt.subplots(figsize=(12, fig_h))

im = ax.imshow(
    df_corr.values,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    interpolation="nearest",
)

# Vertical line at lag=0
zero_x = lags.index(0)
ax.axvline(zero_x - 0.5, color="black", linewidth=1.2, linestyle="--", alpha=0.5)

# Highlight best lag per gauge with a golden border and correlation label
for i, (gauge_id, best_lag) in enumerate(best.items()):
    best_x = lags.index(best_lag)
    rect = mpatches.Rectangle(
        (best_x - 0.5, i - 0.5), 1, 1,
        linewidth=2, edgecolor="#FFD700", facecolor="none"
    )
    ax.add_patch(rect)

    r_val = df_corr.loc[gauge_id, best_lag]
    ax.text(
        best_x, i, f"{r_val:.2f}",
        ha="center", va="center", fontsize=9, fontweight="bold", color="black",
        path_effects=[pe.withStroke(linewidth=2.5, foreground="white")]
    )

# Axes
tick_positions = [i for i, l in enumerate(lags) if l % 5 == 0]
tick_labels = [str(lags[i]) for i in tick_positions]
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels)
ax.set_xlabel("Lag (days)  ←  SFED leads  |  streamflow leads  →")
ax.set_yticks(range(n_gauges))
ax.set_yticklabels(labels, fontsize=10)

# Color y-axis tick labels by river
for tick_label, gauge_id in zip(ax.get_yticklabels(), df_corr.index):
    river = df_lga_upstream[df_lga_upstream["gauge_id"] == gauge_id]["gauge_river"].iloc[0]
    tick_label.set_color(RIVER_COLORS.get(river, DEFAULT_RIVER_COLOR))

season_str = f"months {FLOOD_SEASON_MONTHS[0]}–{FLOOD_SEASON_MONTHS[-1]}"
ax.set_title(
    f"{lga_name} — Spearman r: upstream gauge streamflow vs Floodscan SFED ({FLOODSCAN_VAR})\n"
    f"wet season ({season_str}), SFED ≥ {MIN_SFED}  |  gold border = best lag per gauge"
)

# Legend for river colours
legend_patches = [
    mpatches.Patch(color=color, label=river)
    for river, color in RIVER_COLORS.items()
]
ax.legend(handles=legend_patches, loc="upper right", fontsize=9, title="River")

plt.colorbar(im, ax=ax, label="Spearman r", shrink=0.6)
plt.tight_layout()
plt.show()

## Correlation peakiness: joyplot

Each ridge shows how Spearman r varies across lag days for one gauge (vs Floodscan SFED max). A sharp, narrow peak indicates a well-defined lag; a broad or flat curve means the correlation is similar across many lags.

In [ ]:
lags_arr = np.array(lags)
gauge_list = list(df_corr.index)  # Benue then Niger, sorted by hop_distance
n_gauges = len(gauge_list)

offset_step = 0.55
# First gauge in list → top of plot, matching heatmap order
positions = [(n_gauges - 1 - i) * offset_step for i in range(n_gauges)]

fig, ax = plt.subplots(figsize=(10, max(5, n_gauges * offset_step + 1.5)))

for i, gauge_id in enumerate(gauge_list):
    y_offset = positions[i]
    row = df_corr.loc[gauge_id].values.astype(float)
    river = df_lga_upstream[df_lga_upstream["gauge_id"] == gauge_id]["gauge_river"].iloc[0]
    color = RIVER_COLORS.get(river, DEFAULT_RIVER_COLOR)

    baseline = np.full_like(row, y_offset)
    curve = row + y_offset

    # Positive correlation: filled in river colour
    ax.fill_between(lags_arr, baseline, curve,
                    where=(row >= 0), color=color, alpha=0.5, linewidth=0)
    # Negative correlation: muted grey fill
    ax.fill_between(lags_arr, baseline, curve,
                    where=(row < 0), color="grey", alpha=0.25, linewidth=0)
    # Curve line
    ax.plot(lags_arr, curve, color=color, linewidth=0.9, alpha=0.9)
    # Baseline
    ax.axhline(y_offset, color="black", linewidth=0.3, alpha=0.25, zorder=0)

    # Yellow dot at best lag
    best_lag_i = int(best[gauge_id])
    best_r_i = df_corr.loc[gauge_id, best_lag_i]
    ax.scatter(best_lag_i, best_r_i + y_offset,
               color="#FFD700", edgecolors="black", linewidths=0.8,
               s=40, zorder=5)

# Y-axis: one label per gauge at its baseline position
ax.set_yticks(positions)
ax.set_yticklabels([gauge_label(g) for g in gauge_list], fontsize=8)
for tick_label, gauge_id in zip(ax.get_yticklabels(), gauge_list):
    river = df_lga_upstream[df_lga_upstream["gauge_id"] == gauge_id]["gauge_river"].iloc[0]
    tick_label.set_color(RIVER_COLORS.get(river, DEFAULT_RIVER_COLOR))

# Reference lines
ax.axvline(0, color="black", linewidth=1.2, linestyle="--", alpha=0.5)
ax.set_xlim(MIN_LAG, MAX_LAG)
ax.set_xlabel("Lag (days)  ←  SFED leads  |  streamflow leads  →")

season_str = f"months {FLOOD_SEASON_MONTHS[0]}–{FLOOD_SEASON_MONTHS[-1]}"
ax.set_title(
    f"{lga_name} — correlation peakiness by upstream gauge\n"
    f"Spearman r vs lag  |  Floodscan SFED ({FLOODSCAN_VAR}), wet season ({season_str}), SFED ≥ {MIN_SFED}"
)

legend_patches = [
    mpatches.Patch(color=color, label=river)
    for river, color in RIVER_COLORS.items()
]
ax.legend(handles=legend_patches, loc="upper right", fontsize=9, title="River")

plt.tight_layout()
plt.show()

## Best-correlated gauge: time series

Normalised comparison of the best-correlated gauge's streamflow (at its optimal lag) against Floodscan SFED (max).

In [ ]:
best_gauge_id = df_corr.max(axis=1).idxmax()
best_lag = int(best.loc[best_gauge_id])
best_r = df_corr.loc[best_gauge_id, best_lag]
best_label = gauge_label(best_gauge_id)

df_best = gauge_dfs[best_gauge_id].copy()
df_best["streamflow_lagged"] = df_best["streamflow"].shift(best_lag)

def norm(s):
    rng = s.max() - s.min()
    return (s - s.min()) / rng if rng > 0 else s

# Reindex to a continuous daily range so non-wet-season months appear as gaps
full_range = pd.date_range(df_best["valid_date"].min(), df_best["valid_date"].max(), freq="D")
df_plot = df_best.set_index("valid_date").reindex(full_range)

fig, (ax_ts, ax_sc) = plt.subplots(
    1, 2, figsize=(16, 4), gridspec_kw={"width_ratios": [3, 1]}
)

sfed_label = f"Floodscan SFED ({FLOODSCAN_VAR})"

# Time series — NaN rows from reindex produce natural gaps between flood seasons
ax_ts.plot(df_plot.index, norm(df_plot["sfed"]),
           color="#F2645A", linewidth=0.8, alpha=0.9, label=sfed_label)
ax_ts.plot(df_plot.index, norm(df_plot["streamflow_lagged"]),
           color="#007CE0", linewidth=0.8, alpha=0.7, label=f"{best_label} (lag {best_lag}d)")
ax_ts.set_title(
    f"{lga_name} — best gauge: {best_label}\n"
    f"Spearman r = {best_r:.3f} at lag {best_lag}d  ({sfed_label}, wet season only)"
)
ax_ts.set_ylabel("Normalised value")
ax_ts.legend(loc="upper right", fontsize=9)
ax_ts.xaxis.set_major_locator(mdates.YearLocator(2))
ax_ts.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax_ts.grid(True, alpha=0.3)

# Scatter — raw values, wet-season days only
df_sc = df_best.dropna(subset=["sfed", "streamflow_lagged"])
ax_sc.scatter(df_sc["streamflow_lagged"], df_sc["sfed"],
              color="#007CE0", alpha=0.3, s=10, edgecolors="none")
ax_sc.set_xlabel(f"Streamflow at lag {best_lag}d (m³/s)")
ax_sc.set_ylabel(sfed_label)
ax_sc.set_title(f"Spearman r = {best_r:.3f}")
ax_sc.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## All upstream gauges: scatter plots

Streamflow (at each gauge's best lag) vs Floodscan SFED, coloured by river.

In [ ]:
n_cols = 4
n_rows = (len(df_corr) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.5, n_rows * 3.2))
axes = axes.flatten()

for i, gauge_id in enumerate(df_corr.index):
    ax = axes[i]

    best_lag_i = int(best[gauge_id])
    best_r_i = df_corr.loc[gauge_id, best_lag_i]
    label_i = gauge_label(gauge_id)

    river = df_lga_upstream[df_lga_upstream["gauge_id"] == gauge_id]["gauge_river"].iloc[0]
    color = RIVER_COLORS.get(river, DEFAULT_RIVER_COLOR)

    df_g = gauge_dfs[gauge_id].copy()
    df_g["streamflow_lagged"] = df_g["streamflow"].shift(best_lag_i)
    df_sc = df_g.dropna(subset=["sfed", "streamflow_lagged"])

    ax.scatter(df_sc["streamflow_lagged"], df_sc["sfed"],
               color=color, alpha=0.35, s=10, edgecolors="none")
    ax.set_title(f"{label_i}\nr={best_r_i:.2f}, lag={best_lag_i}d",
                 fontsize=8, color=color)
    ax.set_xlabel("Streamflow (m³/s)", fontsize=7)
    ax.set_ylabel(f"SFED ({FLOODSCAN_VAR})", fontsize=7)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.3)

# Hide unused subplot panels
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

legend_patches = [
    mpatches.Patch(color=color, label=river)
    for river, color in RIVER_COLORS.items()
]
fig.legend(handles=legend_patches, loc="upper right", fontsize=9, title="River",
           bbox_to_anchor=(1.0, 1.0))

season_str = f"months {FLOOD_SEASON_MONTHS[0]}–{FLOOD_SEASON_MONTHS[-1]}"
fig.suptitle(
    f"{lga_name} — all upstream gauges: streamflow vs Floodscan SFED ({FLOODSCAN_VAR})\n"
    f"wet season ({season_str}), SFED ≥ {MIN_SFED}, each at its best lag",
    fontsize=11, y=1.01
)

plt.tight_layout()
plt.show()